## Data Preprocessing CIC-IoT2023

This section covers the preprocessing of the CIC-IoT2023 dataset after extracting its flow-level and packet-level features using the `Feature_extractor_flow_packet_combined.py` script (extraction shown in `GNN4ID.ipynb`) and generation of additional features. Since CIC-IoT2023 dataset is very huge and is very imbalance with having some classes with very low instances, therefore to maintain uniformity we have under and over-sampled data instances. 

In [ ]:
from Utility.Functions import *
import pandas as pd
import glob
import os
from tqdm import tqdm

For our preprocessing (v2, 2026-09), the per-pcap files produced by `GNN4ID.ipynb` (raw NFStream csv +
the rolling features already computed on the whole file) go through three steps, in this order:

1. **`split_csv`** — attacker-MAC filter (paper Table 3 / Sec. 3.2.1), then a **temporal** 80:20 split:
   the latest 20% of each file is the test pool (at most 4,000 rows are taken from it), the earlier 80%
   is the train pool (feature-level duplicates dropped, at most 20,000 rows per file). Nothing is
   written in place: `split/train/<stem>_train.csv`, `split/test/<stem>_test.csv`.
2. **`Combining_classes`** — per class: drop duplicate feature rows, undersample to 20,000 train rows
   **proportionally per sub-attack**, drop every test row whose feature vector equals a train row of
   ANY class, cap the test set at 4,000 (never duplicated), then **oversample the train set only** up to
   20,000 rows (paper Table 4; `oversample=False` keeps the true counts and `class_weights.json` is
   written either way for `train.py --class-weights`).
3. **`build_class8_csvs`** — concatenates the classes, drops the 29 identifier/redundant columns and
   asserts the authors' 97-column header, giving `df_class_8_train.csv` / `df_class_8_test.csv`.

The minority class is still BruteForce (2,336 flows after filtering, 467 in the test set); balancing
happens AFTER the split and only on the train side, so no oversampled row can appear in the test set.


In [ ]:
## Directory that holds the per-pcap csv files (raw/ + features/ written by GNN4ID.ipynb)
directory = 'F:/CIC_IOT/Extracted_Flow_Features/'
List_of_CSV_File = glob.glob(os.path.join(directory, 'features', '*.csv'))   # rolling features already inside


### Dataset Division & Filtering

For the attack class samples, we are specifically filtering the data by retaining only those flow instances where the attacker's MAC address appears as either the source or destination. This targeted approach ensures that non-attack flows are excluded from the analysis, focusing solely on the relevant attack data.

Similarly, we are removing the samples with attacker's MAC address for the Benign Samples so that our data is more clean and filtered.

Furthermore, we are capping the number of samples to enhance the efficiency and manageability of the dataset. The test dataset is limited to a maximum of 4,000 samples, while the training dataset is restricted to 20,000 samples. This strategic sampling allows for effective model training and evaluation without compromising on performance or computational resources.

The provided code offers full flexibility for personalization, allowing you to process a specific number of files or focus on particular attack classes based on your requirements. Whether you need to analyze just a subset of the data or target specific attack types, the code can be tailored to meet your needs.


### Test & Train Split

In [ ]:
# MAC filter + temporal 80/20 split + per-file caps, one call per pcap csv (nothing overwritten)
for files in tqdm(List_of_CSV_File):
    split_csv(files, out_dir=os.path.join(directory, 'split'))


##### Combining Sub-Classes into Broad Classes

In [ ]:
Attack_Classes = ['Benign', 'BruteForce', 'DDos', 'Dos', 'Mirai', 'Recon', 'Spoofing', 'WebBased']
label_dict = {'Benign': 0, 'WebBased': 1, 'Spoofing': 2, 'Recon': 3, 'Mirai': 4, 'Dos': 5, 'DDos': 6, 'BruteForce': 7}
## Combine the sub-attack files of each class: dedup, proportional caps (20k train / 4k test),
## global test-vs-train removal, then oversample the TRAIN side only (paper Table 4).
report = Combining_classes(os.path.join(directory, 'split'), Attack_Classes, label_dict=label_dict,
                           oversample=True, out_dir=os.path.join(directory, 'combined'))


#### Transforming Data into Single Train and Test File

In [ ]:
## One train file + one test file: concat the classes, drop the 29 identifier / redundant columns
## (the old cells 11-14) and assert the authors' 97-column header.
result = build_class8_csvs(os.path.join(directory, 'combined'), directory)
result
